## Naive Bayes (the easy way)

Vamos "trapacear" usando o sklearn.naive_bayes para treinar um classificador de spam! A maior parte do código é só carregar nossos dados de treino em um DataFrame do pandas com o qual podemos brincar:

In [ ]:
import os
import io
import pandas as pd
from pandas import DataFrame
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

def readFiles(path): # essa funcao vai ler todos os arquivos contidos na pasta especificada pelo path
    for root, dirnames, filenames in os.walk(path):
        for filename in filenames:
            path = os.path.join(root, filename)
            inBody = False
            lines = []
            f = io.open(path, 'r', encoding='latin1')
            for line in f:
                if inBody:
                    lines.append(line)
                elif line == '\n':
                    inBody = True
            f.close()
            message = '\n'.join(lines)
            yield path, message

def dataFrameFromDirectory(path, classification): # define uma funcao que le os arquivos para organizar num Dataframe
    rows = []
    index = []
    for filename, message in readFiles(path):
        rows.append({'message': message, 'class': classification})
        index.append(filename)
    return DataFrame(rows, index=index)

data = DataFrame({'message': [], 'class': []})#cria um DataFrame inicial vazio com duas colunas

data = pd.concat([data, dataFrameFromDirectory(
    'C:/Users/zinho/OneDrive/Documentos/Lamia/Card 13 - Predição e a Base de Aprendizado de Máquina (II)/emails/spam',
    'spam')])
#concatena os spam e os ham dentro de data
data = pd.concat([data, dataFrameFromDirectory(
    'C:/Users/zinho/OneDrive/Documentos/Lamia/Card 13 - Predição e a Base de Aprendizado de Máquina (II)/emails/ham',
    'ham')])

In [ ]:
data.head()

Agora vamos usar um CountVectorizer para dividir cada mensagem na sua lista de palavras, e jogar isso em um classificador MultinomialNB. Basta chamar fit() e já temos um filtro de spam treinado e pronto para usar! É simples assim.

In [ ]:
vectorizer = CountVectorizer()
counts = vectorizer.fit_transform(data['message'].values) # aprende as palavras e transforma o texto do email em um vetor numerico

classifier = MultinomialNB() #instancia o classificador Naive Bayes
targets = data['class'].values
classifier.fit(counts, targets) #treina o modelo

Vamos testar:

In [ ]:
examples = ['Free Viagra now!!!', "Hi Bob, how about a game of golf tomorrow?"] 
example_counts = vectorizer.transform(examples) 
predictions = classifier.predict(example_counts) 
predictions 

## Atividade

Nosso conjunto de dados é pequeno, então nosso classificador de spam não é tão bom assim de fato. Tente rodar alguns e-mails de teste diferentes nele e veja se você obtém os resultados esperados.

Se você realmente quiser se desafiar, tente aplicar train/test a esse classificador de spam - veja o quão bem ele consegue prever um subconjunto dos e-mails ham e spam.

In [ ]:
novos_emails = [
    "Lowest rates available for term life insurance! Take a moment and fill out our online form to see the low rate you qualify for. Save up to 70% from regular rates! Smokers accepted!", # spam obvio
    "Hey boss, I'll be 10 minutes late to the meeting today.", # ham normal
    "Cheap loans and fast approval, no credit check required!", # spam obvio
    "Can we reschedule our lunch for tomorrow at 12:30?" # ham normal
]

#transformando o texto em números usando o vectorizer já treinado
novos_counts = vectorizer.transform(novos_emails)

#fazendo a previsão
novas_previsoes = classifier.predict(novos_counts)

#imprimindo os resultados
for email, previsao in zip(novos_emails, novas_previsoes):
    print(f"[{previsao.upper()}] - {email}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

#dividindo os dados (80% treino, 20% teste)
X_train, X_test, y_train, y_test = train_test_split(data['message'].values, data['class'].values, test_size=0.2, random_state=42)

#vetorizando os dados
vectorizer_tt = CountVectorizer()
counts_train = vectorizer_tt.fit_transform(X_train)
counts_test = vectorizer_tt.transform(X_test) # O modelo não pode aprender palavras novas do teste

#treinando o classificador
classifier_tt = MultinomialNB()
classifier_tt.fit(counts_train, y_train)

#fazendo as previsões com os dados de teste (que o modelo nunca viu)
predicoes_teste = classifier_tt.predict(counts_test)

#avaliando e imprimindo a precisão (Accuracy)
precisao = accuracy_score(y_test, predicoes_teste)
print(f"precisao do modelo nos dados de teste: {precisao * 100:.2f}%")